# 03 · El exponente de Medellín

*Segundos contra vidas* · Hackathon de Datos 2026 · ITM

**Pregunta:** si la velocidad de un corredor sube un 10%, ¿cuánto suben los incidentes graves?

El modelo de Nilsson dice que los siniestros crecen como una potencia de la velocidad: **graves ∝ velocidad^β**.
β es el *exponente*. Aquí lo estimamos con datos de Medellín, con una **regresión binomial negativa**:
- unidad: corredor × hora × año (12 corredores de Google Traffic, 6:00–20:00, 2017–2019, D4);
- respuesta: incidentes graves (herido + muerto) de la base del ITM (D8), asignados al corredor **por coordenadas**;
- exposición: volumen de vehículos de cada corredor y hora (SIMM);
- controles: corredor y año (y franja u hora, D11).

Cada parte: código → conclusión → decisión.

## 0 · Configuración

In [ ]:
import os, sys, unicodedata, re, warnings
from pathlib import Path
from statsmodels.tools.sm_exceptions import ConvergenceWarning, HessianInversionWarning
for w in (ConvergenceWarning, HessianInversionWarning):
    warnings.filterwarnings("ignore", category=w)                # la convergencia se revisa en ajustar()

raiz = Path.cwd()
while not (raiz / "src" / "fuentes.py").exists():
    if raiz.parent == raiz:
        raise FileNotFoundError(f"No encontré el proyecto. El notebook está en: {Path.cwd()}")
    raiz = raiz.parent
os.chdir(raiz)
sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats
from scipy.spatial import cKDTree
from pyproj import Transformer
from src.descarga import leer
from src.velocidad import CORREDORES, tiempo_viaje_gt

# Los alias deben ser los confirmados por Johan (rama johan, commit "Alias en formato ITM confirmados con coordenadas")
assert CORREDORES["AUT_SUR"][1].startswith("CR 50"), "src/velocidad.py está desactualizado: une la rama johan y haz git pull"

pd.set_option("display.max_columns", 40)
Path("data/processed").mkdir(parents=True, exist_ok=True)
RAPIDAS = ["AV_REGIONAL", "AUT_NORTE", "AUT_SUR", "AV_PALMAS"]          # D18
NOMBRE = {cid: v[0] for cid, v in CORREDORES.items()}
print("Proyecto:", raiz)

In [ ]:
# Nombres de columnas uniformes: minúscula, sin tildes, con guion bajo
def _arreglar_texto(s):
    for codif in ("latin-1", "cp1252"):
        try:
            return s.encode(codif).decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            pass
    return s

def normalizar_columnas(df):
    nuevas = []
    for c in df.columns:
        c = unicodedata.normalize("NFKD", _arreglar_texto(str(c))).encode("ascii", "ignore").decode()
        nuevas.append(re.sub(r"[^0-9a-z]+", "_", c.lower()).strip("_"))
    df.columns = nuevas
    return df

def numero(serie):
    return pd.to_numeric(serie.astype(str).str.replace(",", ".", regex=False), errors="coerce")

def en_medellin(lat, lon):                         # D10: lo de fuera de la ciudad es faltante
    return (lat > 6.0) & (lat < 6.5) & (lon > -75.8) & (lon < -75.3)

## 1 · Tabla maestra (corredor × hora × año)
Cambios frente a la versión provisional:
- Corredores con el `corredor_id` del catálogo de Johan.
- **Asignación por coordenadas:** un incidente es del corredor si su sensor del SIMM más cercano (de cualquiera de los 49 corredores) es de ese corredor y está a ≤ 100 m. Así se ubican la Regional y Las Palmas, y se toma solo el tramo con sensores.
- Los alias por dirección quedan como **validación cruzada**.
- Velocidad con `tiempo_viaje_gt()` de Johan (tramos por id, no por nombre).

#### 1.1 Incidentes graves de la base del ITM (2017–2019, 6:00–20:00)

In [ ]:
itm = normalizar_columnas(leer("incidentes_itm"))
itm["ano"] = pd.to_numeric(itm["ano"], errors="coerce")
itm["hora"] = pd.to_datetime(itm["hora_incidente"].astype(str), format="mixed", errors="coerce").dt.hour

graves = itm[itm["gravedad_incidente"].isin(["HERIDO", "MUERTO"])
             & itm["ano"].between(2017, 2019) & itm["hora"].between(6, 20)].copy()
graves["ano"] = graves["ano"].astype(int)
graves["lat"], graves["lon"] = numero(graves["latitud"]), numero(graves["longitud"])
graves["coord_ok"] = en_medellin(graves["lat"], graves["lon"])
graves = graves.reset_index(drop=True)

print(f"Incidentes graves 2017–2019, 6:00–20:00: {len(graves):,}")
print(f"Con coordenadas válidas en Medellín: {graves['coord_ok'].mean() * 100:.1f}%")
print("Hora sin leer (%):", round(itm["hora"].isna().mean() * 100, 2))

#### 1.2 Asignar incidentes a corredores por coordenadas

In [ ]:
simm = normalizar_columnas(leer("velocidad_simm"))
simm["corredor"] = simm["corredor"].astype(str).str.strip()
simm_a_id = {v[2]: cid for cid, v in CORREDORES.items()}

sens = simm.assign(x=numero(simm["longitud"]), y=numero(simm["latitud"]))
sens = sens.dropna(subset=["x", "y"]).drop_duplicates(["corredor", "carril"]).reset_index(drop=True)
sens["corredor_id"] = sens["corredor"].map(simm_a_id)          # NaN = corredor del SIMM fuera de los 12
print(f"Sensores: {len(sens)} de {sens['corredor'].nunique()} corredores del SIMM · "
      f"de los 12: {sens['corredor_id'].notna().sum()}")

ok = graves["coord_ok"]
x, y = Transformer.from_crs(4326, 3116, always_xy=True).transform(graves.loc[ok, "lon"].values, graves.loc[ok, "lat"].values)
dist, idx = cKDTree(sens[["x", "y"]].values).query(np.column_stack([x, y]), k=min(15, len(sens)))

cor_vecino = sens["corredor"].to_numpy(dtype=object)[idx]                      # corredor del SIMM de cada vecino
graves.loc[ok, "dist_m"] = dist[:, 0]
graves.loc[ok, "simm_cercano"] = cor_vecino[:, 0]
# Distancia al sensor más cercano de OTRO corredor (para los casos ambiguos)
otro = cor_vecino != cor_vecino[:, [0]]
pos = np.where(otro.any(1), otro.argmax(1), 0)
graves.loc[ok, "dist_otro_m"] = np.where(otro.any(1), dist[np.arange(len(pos)), pos], np.inf)
graves.loc[ok, "simm_otro"] = np.where(otro.any(1), cor_vecino[np.arange(len(pos)), pos], None)
graves["corredor_id"] = graves["simm_cercano"].map(simm_a_id)

for r in (30, 50, 100):
    graves[f"cor_{r}m"] = graves["corredor_id"].where(graves["dist_m"] <= r)

resumen = pd.DataFrame({f"≤{r} m": graves[f"cor_{r}m"].value_counts() for r in (30, 50, 100)}) \
            .reindex(list(CORREDORES)).fillna(0).astype(int)
resumen.index = resumen.index.map(NOMBRE)
print(f"Asignados a ≤100 m: {graves['cor_100m'].notna().sum():,} "
      f"({graves['cor_100m'].notna().mean() * 100:.1f}% de los graves)")
resumen

#### 1.3 Casos ambiguos
Incidentes asignados a ≤ 100 m que también tienen un sensor de **otro** corredor a ≤ 100 m (por ejemplo, Regional y Autopista Norte, que corren paralelas). Se quedan con el más cercano; aquí se mide cuántos son.

In [ ]:
a = graves[graves["cor_100m"].notna()]
amb = a[a["dist_otro_m"] <= 100]
print(f"Ambiguos: {len(amb):,} de {len(a):,} ({len(amb) / len(a) * 100:.1f}%)\n")
print("Pares más frecuentes (asignado → el otro corredor cercano):")
print(amb.assign(asignado=amb["cor_100m"].map(NOMBRE)).groupby(["asignado", "simm_otro"]).size()
         .sort_values(ascending=False).head(8).to_string())
tab_amb = pd.DataFrame({"asignados": a.groupby("cor_100m").size(), "ambiguos": amb.groupby("cor_100m").size()}) \
            .reindex(list(CORREDORES)).fillna(0).astype(int)
tab_amb["% ambiguos"] = (tab_amb["ambiguos"] / tab_amb["asignados"].replace(0, np.nan) * 100).round(1)
tab_amb.index = tab_amb.index.map(NOMBRE)
tab_amb

#### 1.4 Validación cruzada con los alias de Johan
¿Qué tanto coincide la dirección del incidente con el alias del corredor asignado por coordenadas? En la Regional y Las Palmas se espera poco (no tienen número propio).

In [ ]:
def coincide_alias(direccion, cid):
    partes = [p.strip() for p in str(direccion).upper().split(" CON ")]
    if cid == "AV_PALMAS":
        return any("PALMAS" in p for p in partes)
    patron = re.compile(CORREDORES[cid][1])
    return any(patron.fullmatch(p) for p in partes)

a = graves[graves["cor_100m"].notna()].copy()
a["alias_ok"] = [coincide_alias(d, c) for d, c in zip(a["direccion_interpolada"], a["cor_100m"])]
val = a.groupby("cor_100m")["alias_ok"].agg(["size", "mean"]).reindex(list(CORREDORES))
val.columns = ["asignados ≤100 m", "% con el alias en la dirección"]
val["% con el alias en la dirección"] = (val["% con el alias en la dirección"] * 100).round(1)
val.index = val.index.map(NOMBRE)
val

#### 1.5 Velocidad por corredor, hora y año (Google Traffic, `tiempo_viaje_gt()`)

In [ ]:
tv = tiempo_viaje_gt()
tv["ano"] = pd.to_datetime(tv["fecha"]).dt.year
tv = tv[tv["ano"].between(2017, 2019) & tv["hora"].between(6, 20) & (tv["velocidad_gt_kmh"] > 0)]
vel = tv.groupby(["corredor_id", "hora", "ano"])["velocidad_gt_kmh"].mean().rename("vel").reset_index()
print("Corredores con velocidad:", vel["corredor_id"].nunique(), "de", len(CORREDORES))
print("Faltan:", set(CORREDORES) - set(vel["corredor_id"]) or "ninguno")

#### 1.6 Exposición: volumen por corredor y hora (SIMM)
El SIMM solo trae jul–ago 2020 (D21). Se usa el **perfil horario** de volumen de cada corredor, suponiendo que la forma del día se mantiene. Lo que sea constante por corredor lo absorbe el efecto fijo.

In [ ]:
s = simm[(simm["velocidad_km_h"] > 0) & (simm["velocidad_km_h"] < 150) & (simm["intensidad"] > 0)]
s = s.assign(corredor_id=s["corredor"].map(simm_a_id)).dropna(subset=["corredor_id"])
vol = (s[s["hora"].between(6, 20)].groupby(["corredor_id", "hora"])["intensidad"].mean()
         .rename("vol").reset_index())
print("Corredores con volumen:", vol["corredor_id"].nunique(), "de", len(CORREDORES))
print("Sin volumen:", set(CORREDORES) - set(vol["corredor_id"]) or "ninguno")

#### 1.7 Unir todo

In [ ]:
def contar(col):
    return graves.dropna(subset=[col]).groupby([col, "hora", "ano"]).size() \
                 .rename_axis(["corredor_id", "hora", "ano"])

tabla = vel.merge(vol, on=["corredor_id", "hora"], how="inner")
for r, nombre in [(100, "graves"), (50, "graves_50m"), (30, "graves_30m")]:
    tabla = tabla.merge(contar(f"cor_{r}m").rename(nombre).reset_index(), on=["corredor_id", "hora", "ano"], how="left")
    tabla[nombre] = tabla[nombre].fillna(0).astype(int)

tabla["log_vel"] = np.log(tabla["vel"])
tabla["franja"] = pd.cut(tabla["hora"], [5, 8, 16, 20], labels=["pico_am", "valle", "pico_pm"])
tabla["tipo"] = np.where(tabla["corredor_id"].isin(RAPIDAS), "rapida", "arteria")
tabla.to_csv("data/processed/tabla_maestra.csv", index=False)

print(f"Filas: {len(tabla)} (corredores: {tabla['corredor_id'].nunique()}, horas: {tabla['hora'].nunique()}, años: {tabla['ano'].nunique()})")
print(f"Incidentes graves en la tabla: {tabla['graves'].sum():,} · celdas con 0: {(tabla['graves'] == 0).mean() * 100:.1f}%")
print(f"Media {tabla['graves'].mean():.1f} · varianza {tabla['graves'].var():.1f} (si la varianza >> media: binomial negativa)")
tabla.head()

### Conclusión
- _(completar después de correr)_

### Decisión
- _(completar)_

## 2 · ¿Se ve la relación a simple vista?
Tasa = graves / volumen. Izquierda: todos los corredores. Derecha: **dentro de cada corredor** (se resta el promedio de cada uno), que es la comparación que hace el modelo.

In [ ]:
t = tabla.copy()
t["log_tasa"] = np.log((t["graves"] + 0.5) / t["vol"])
t["log_vel_c"] = t["log_vel"] - t.groupby("corredor_id")["log_vel"].transform("mean")
t["log_tasa_c"] = t["log_tasa"] - t.groupby("corredor_id")["log_tasa"].transform("mean")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
for c, d in t.groupby("corredor_id"):
    ax1.scatter(d["vel"], d["log_tasa"], s=12, alpha=0.6, label=NOMBRE[c])
ax1.set_xscale("log"); ax1.set_xlabel("Velocidad media (km/h, escala log)"); ax1.set_ylabel("log(graves / volumen)")
ax1.set_title("Entre corredores"); ax1.legend(fontsize=7, ncol=2)

ax2.scatter(t["log_vel_c"], t["log_tasa_c"], s=12, alpha=0.5, color="#2c3e50")
pend, inter = np.polyfit(t["log_vel_c"], t["log_tasa_c"], 1)
xs = np.linspace(t["log_vel_c"].min(), t["log_vel_c"].max(), 50)
ax2.plot(xs, inter + pend * xs, color="#c0392b", lw=2, label=f"pendiente ≈ {pend:.2f}")
ax2.axhline(0, color="grey", lw=0.5); ax2.axvline(0, color="grey", lw=0.5)
ax2.set_xlabel("log(velocidad) respecto al promedio del corredor")
ax2.set_ylabel("log(tasa) respecto al promedio del corredor")
ax2.set_title("Dentro de cada corredor"); ax2.legend()
plt.tight_layout(); plt.show()

### Conclusión
- _(completar después de correr)_

### Decisión
- _(completar)_

## 3 · El modelo
$\log E[\text{graves}] = \log(\text{volumen}) + \beta \cdot \log(\text{velocidad}) + \text{corredor} + \text{año}$

β es el exponente: si la velocidad sube 10%, los graves se multiplican por $1{,}1^\beta$.

⚠ Dentro de un corredor, las horas rápidas suelen ser las de menos congestión (y menos conflicto). Por eso β puede salir bajo o negativo: no sería que la velocidad proteja, sino que la hora y la congestión se mezclan con ella.

#### 3.1 ¿Poisson o binomial negativa?

In [ ]:
def ajustar(formula, d, y="graves", avisar=True):
    """Binomial negativa que arranca desde la Poisson. Si no converge, Poisson con errores robustos."""
    f = formula.replace("graves", y, 1)
    pois = smf.poisson(f, data=d, exposure=d["vol"]).fit(disp=0, maxiter=300)
    try:
        r = smf.negativebinomial(f, data=d, exposure=d["vol"]).fit(
            start_params=np.append(pois.params, 0.1), disp=0, maxiter=500)
        if r.mle_retvals.get("converged") and np.isfinite(r.bse).all():
            return r
    except Exception:
        pass
    if avisar:
        print(f"⚠ La binomial negativa no convergió ({f}): se usa Poisson con errores robustos")
    return smf.poisson(f, data=d, exposure=d["vol"]).fit(disp=0, maxiter=300, cov_type="HC0")

f_A = "graves ~ log_vel + C(corredor_id) + C(ano)"
pois = smf.poisson(f_A, data=tabla, exposure=tabla["vol"]).fit(disp=0, maxiter=200)
nb_A = ajustar(f_A, tabla)

lr = 2 * (nb_A.llf - pois.llf)
print("Modelo ajustado:", type(nb_A.model).__name__)
print(f"alpha (sobredispersión): {nb_A.params.get('alpha', 0):.3f}")
print(f"Prueba de razón de verosimilitud: LR = {lr:.1f}, p = {stats.chi2.sf(lr, 1) / 2:.2g}")
print(f"AIC Poisson: {pois.aic:.0f} · AIC binomial negativa: {nb_A.aic:.0f} (menor = mejor)")

#### 3.2 El exponente
- **Modelo A:** corredor + año.
- **Modelo B:** + franja (pico_am 6–8, valle 9–16, pico_pm 17–20; las mismas del costo en tiempo).
- **Modelo C:** + hora exacta (D11). Es el más exigente: solo usa cómo cambia la velocidad de *cada* corredor respecto al patrón horario común.

In [ ]:
f_B = f_A + " + C(franja)"
f_C = f_A + " + C(hora)"
nb_B, nb_C = ajustar(f_B, tabla), ajustar(f_C, tabla)

def fila(nombre, res, var="log_vel"):
    b, (lo, hi) = res.params[var], res.conf_int().loc[var]
    return {"modelo": nombre, "beta": b, "IC95_inf": lo, "IC95_sup": hi, "p": res.pvalues[var],
            "+10% velocidad → graves": f"{(1.1 ** b - 1) * 100:+.1f}%", "AIC": res.aic}

exponente = pd.DataFrame([fila("A · corredor + año", nb_A), fila("B · + franja", nb_B),
                          fila("C · + hora (D11)", nb_C)]).set_index("modelo")
exponente.round(3)

#### 3.3 ¿Es distinto en las vías rápidas? (D18)
**Modelo D:** un β para las vías rápidas y otro para las arterias, con los mismos controles del modelo A.

In [ ]:
f_D = "graves ~ log_vel:C(tipo) + C(corredor_id) + C(ano)"
nb_D = ajustar(f_D, tabla)
v_rap, v_art = "log_vel:C(tipo)[rapida]", "log_vel:C(tipo)[arteria]"
dif = nb_D.params[v_rap] - nb_D.params[v_art]
ee = np.sqrt(nb_D.cov_params().loc[v_rap, v_rap] + nb_D.cov_params().loc[v_art, v_art]
             - 2 * nb_D.cov_params().loc[v_rap, v_art])
print(f"¿Difieren? diferencia = {dif:.2f}, p = {2 * stats.norm.sf(abs(dif / ee)):.3f}")
pd.DataFrame([fila("D · vías rápidas", nb_D, v_rap), fila("D · arterias", nb_D, v_art)]).set_index("modelo").round(3)

#### 3.4 Comparación con la literatura

In [ ]:
ref = pd.DataFrame({
    "fuente": ["Medellín · modelo A", "Medellín · modelo B", "Medellín · modelo C", "Medellín · D vías rápidas",
               "Nilsson (siniestros con heridos)", "SWOV urbano (rango documentado)"],
    "beta": [nb_A.params["log_vel"], nb_B.params["log_vel"], nb_C.params["log_vel"], nb_D.params[v_rap], 2.0, 3.0],
    "inf": [*exponente["IC95_inf"], nb_D.conf_int().loc[v_rap, 0], 2.0, -0.5],
    "sup": [*exponente["IC95_sup"], nb_D.conf_int().loc[v_rap, 1], 2.0, 6.5],
})
fig, ax = plt.subplots(figsize=(9, 3.8))
colores = ["#c0392b"] * 4 + ["#7f8c8d"] * 2
for i, r in ref.iterrows():
    ax.plot([r["inf"], r["sup"]], [i, i], color=colores[i], lw=3, alpha=0.5)
    ax.scatter(r["beta"], i, color=colores[i], s=60, zorder=3)
    ax.text(r["sup"] + 0.15, i, f"{r['beta']:.2f}", va="center", fontsize=9)
ax.axvline(0, ls="--", color="black", lw=1)
ax.set_yticks(range(len(ref)), ref["fuente"]); ax.invert_yaxis()
ax.set_xlabel("Exponente β (con IC 95%)")
ax.set_title("El exponente de Medellín frente a la literatura")
plt.tight_layout(); plt.show()

### Conclusión
- _(completar después de correr)_

### Decisión
- _(completar)_

## 4 · ¿Qué tan estable es el exponente?

#### 4.1 Bootstrap por corredor
Con solo 12 corredores, el intervalo del modelo puede quedarse corto. Se re-sortean corredores con reemplazo 500 veces y se re-estima β. Esta distribución es la que usará el Monte Carlo.

In [ ]:
rng = np.random.default_rng(2026)
corredores = tabla["corredor_id"].unique()
betas = []
for _ in range(500):
    muestra = rng.choice(corredores, size=len(corredores), replace=True)
    bs = pd.concat([tabla[tabla["corredor_id"] == c].assign(corredor_id=f"{c}_{k}") for k, c in enumerate(muestra)])
    try:
        betas.append(ajustar(f_A, bs, avisar=False).params["log_vel"])
    except Exception:
        pass
betas = np.array(betas)
np.save("data/processed/betas_bootstrap.npy", betas)

lo, hi = np.percentile(betas, [2.5, 97.5])
print(f"Re-estimaciones válidas: {len(betas)} de 500")
print(f"β bootstrap: mediana {np.median(betas):.2f} · IC 95% [{lo:.2f}, {hi:.2f}]")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(betas, bins=40, color="#2c3e50", alpha=0.8)
ax.axvline(nb_A.params["log_vel"], color="#c0392b", lw=2, label="β del modelo A")
ax.axvline(2.0, color="#7f8c8d", ls="--", lw=2, label="Nilsson (2)")
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("β"); ax.set_title("Distribución del exponente (bootstrap por corredor)"); ax.legend()
plt.tight_layout(); plt.show()

#### 4.2 Dejando un corredor afuera
¿Algún corredor solo determina el resultado?

In [ ]:
loo = []
for c in corredores:
    d = tabla[tabla["corredor_id"] != c]
    r = ajustar(f_A, d)
    loo.append({"sin": NOMBRE[c], "beta": r.params["log_vel"],
                "inf": r.conf_int().loc["log_vel", 0], "sup": r.conf_int().loc["log_vel", 1]})
loo = pd.DataFrame(loo).sort_values("beta")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.errorbar(loo["beta"], range(len(loo)), xerr=[loo["beta"] - loo["inf"], loo["sup"] - loo["beta"]],
            fmt="o", color="#2c3e50", ecolor="#95a5a6", capsize=3)
ax.axvline(nb_A.params["log_vel"], color="#c0392b", lw=2, label="Con todos")
ax.axvline(0, color="black", lw=1)
ax.set_yticks(range(len(loo)), "sin " + loo["sin"])
ax.set_xlabel("β"); ax.set_title("Exponente quitando un corredor a la vez"); ax.legend()
plt.tight_layout(); plt.show()

#### 4.3 ¿Depende del radio de asignación?
El mismo modelo A contando los incidentes a ≤ 30 m, ≤ 50 m y ≤ 100 m del sensor.

In [ ]:
radios = pd.DataFrame([fila(f"A · ≤{r} m", ajustar(f_A, tabla, y)) for r, y in
                       [(30, "graves_30m"), (50, "graves_50m"), (100, "graves")]]).set_index("modelo")
radios.round(3)

### Conclusión
- _(completar después de correr)_

### Decisión
- _(completar)_

## 5 · ¿Qué significa para las vías rápidas?
Reducción esperada de incidentes graves si la velocidad media baja a 50 km/h donde hoy la supera (solo como lectura del exponente; el cálculo completo, hora por hora y con incertidumbre, es el Monte Carlo).

In [ ]:
b_mod, b_lo, b_hi = nb_A.params["log_vel"], np.percentile(betas, 2.5), np.percentile(betas, 97.5)

def reduccion(d, b):
    peso = d["graves"] + 0.5
    return (1 - np.average((np.minimum(d["vel"], 50) / d["vel"]) ** b, weights=peso)) * 100

filas = []
for c, d in tabla[tabla["corredor_id"].isin(RAPIDAS)].groupby("corredor_id"):
    filas.append({"corredor": NOMBRE[c], "vel_media": d["vel"].mean(), "% horas sobre 50": (d["vel"] > 50).mean() * 100,
                  "graves_por_año": d["graves"].sum() / d["ano"].nunique(),
                  "reducción_Medellín_%": reduccion(d, b_mod),
                  "IC_bootstrap_%": f"{reduccion(d, b_lo):.1f} a {reduccion(d, b_hi):.1f}",
                  "reducción_Nilsson_%": reduccion(d, 2.0)})
lect = pd.DataFrame(filas).set_index("corredor")
lect.round(2)

### Conclusión
- _(completar después de correr)_

### Decisión
- _(completar)_

## 6 · Resumen
_(completar al final)_

- Qué es el exponente y cómo se estimó.
- Cuánto vale en Medellín y qué tan seguro es (IC del modelo y del bootstrap).
- Si es distinto en las vías rápidas (modelo D).
- Cómo se compara con Nilsson y con la literatura.
- Qué valor se usará en el Monte Carlo y por qué.
- Limitaciones: asignación por cercanía a sensores (no por la línea del corredor), volumen de 2020, velocidad media del tramo, solo 6:00–20:00, la hora se confunde con la congestión.